# surfaces

> Containers and small pieces of visual information

In [ ]:
#| default_exp surfaces

Surfaces group content without prescribing what it contains. A card is a visible boundary around a group of children, using the colour, spacing, and radius tokens already in place. A badge is a short label: a category, a count, or a status. An avatar identifies a person or team, by image or by initials.

In [ ]:
from answerui.registry import reg, hypelt
from answerui.layout import Stack, Cluster
from answerui.demo import example
from fasthtml.common import H3, P, Span, Img
import answerui.base  # the reset and tokens this styling is written on
import answerui.controls  # the button rules the avatar button overrides

In [ ]:
from fasthtml.common import Article, Header, Footer, Small, to_xml
from answerui.layout import Grid
from answerui.controls import Button
from answerui.demo import demo
from urllib.parse import quote

## Card

We style a card a surface with a hairline border and a raised shadow to set it just above the page. It has no rims/bevels, because they imply pressable. The `[card]` attribute supplies that treatment. `Card(...)` adds it to a stack, giving the children a sensible gap without adding a wrapper.

In [ ]:
_card_css = r"""
[card] {
  --face: var(--n-2); background: var(--face); border: 1px solid var(--n-1); box-shadow: var(--raised);
  border-radius: var(--radius); padding: var(--space);
}
"""

@hypelt(sel='[card]', layer='components', css=_card_css)
def Card(
    *c,       # Content of the card
    gap=None, # Space between children; None uses --space
    **attrs,  # Attributes for the containing div
):
    "A bordered surface with vertically arranged content"
    return Stack(*c, gap=gap, card=True, **attrs)

The card owns its padding and the gap between these children; the surrounding layout will own the space between cards.

In [ ]:
@example(Card)
def _card_example():
    return Card(H3('Shared workspace'), P('Keep your projects and notes together in one place.'))

In [ ]:
Card.demo()

```html
<div stack card>
  <h3>Shared workspace</h3>
  <p>Keep your projects and notes together in one place.</p>
</div>

```

### Composition

The surface does not require a particular layout or set of sections. Here a grid places two cards side by side when there is room. The second is a self-contained article with a header, body, and footer. It carries `card` and `stack` directly, and its header and footer choose their own layouts. There are no special card-section constructors.

In [ ]:
demo(Grid(
    _card_example(),
    Article(
        Header(H3('Field notes'), Small('Updated today'), stack=True, style='--gap: var(--space-3)'),
        P('Observations and ideas from the team, ready for the next meeting.'),
        Footer(Button('Open project', type='button'), Button('Archive', type='button', secondary=True), cluster=True),
        card=True, stack=True)))

```html
<div grid>
  <div stack card>
    <h3>Shared workspace</h3>
    <p>Keep your projects and notes together in one place.</p>
  </div>
  <article card stack>
    <header stack style="--gap: var(--space-3)">
      <h3>Field notes</h3>
<small>Updated today</small>    </header>
    <p>Observations and ideas from the team, ready for the next meeting.</p>
    <footer cluster>
<button type="button">Open project</button><button type="button" secondary>Archive</button>    </footer>
  </article>
</div>

```

### Customisation

The card uses the shared tokens directly. Changing `--radius` or `--space` changes its defaults along with the rest of the interface. To change cards alone, register ordinary CSS in the app layer, such as `[card] { padding: var(--space1); }`. There is no second set of card tokens to keep in sync. A single card can use `style`, while `gap` remains the stack option:

In [ ]:
to_xml(Card(P('A compact note'), gap='var(--space-2)', style='padding: var(--space-2)'))

'<div stack card style="--gap: var(--space-2);padding: var(--space-2)">\n  <p>A compact note</p>\n</div>\n'

The surface and layout attributes are independent. For example, `Grid(..., card=True)` puts a grid inside the same boundary without a nested card wrapper. Bare articles, sections, and divs remain unboxed unless they opt in.

## Badge

A badge is a short label: a category, a count, or a status. It uses a soft accent background and strong accent text, leaving the solid, raised treatment to buttons. `secondary` gives it a neutral treatment.

In [ ]:
_badge_css = r"""
[badge] {
  display: inline-block; padding: .125em .5em;
  font-size: var(--text-1); font-weight: var(--weight1); line-height: var(--leading-1);
  border-radius: var(--radius-1); background: var(--v-3); color: var(--v2);
}
"""

@hypelt(sel='[badge]', layer='components', css=_badge_css)
def Badge(*c, **attrs):
    "A small text label, optionally coloured by a severity attribute"
    return Span(*c, badge=True, **attrs)

Badge.variant('secondary', 'background: var(--n-1); color: var(--n2);', layer='components', doc='A neutral label.')

Badge.secondary [components] :is([badge])[secondary]: A neutral label.

The padding follows the text size, keeping the label compact. The words carry the meaning; colour reinforces it. The badge adds no interaction or live-region role.

In [ ]:
@example(Badge)
def _badge_example():
    return Cluster(Badge('New'), Badge('Draft', secondary=True), Badge('Active', success=True),
        Badge('Pending', warning=True), Badge('Failed', danger=True), Badge('Preview', info=True),
        gap='var(--space-2)')

In [ ]:
Badge.demo()

```html
<div cluster style="--gap: var(--space-2);">
<span badge>New</span><span badge secondary>Draft</span><span badge success>Active</span><span badge warning>Pending</span><span badge danger>Failed</span><span badge info>Preview</span></div>

```

Badges have no outer margin. Here the header places one beside its heading, and the card supplies the space before the description. A count is just another text child, not content generated by CSS:

In [ ]:
demo(Card(
    Cluster(H3('Field notes'), Badge('Shared'), gap='var(--space-2)'),
    P('Observations and ideas from the team.'),
    Cluster(Span('Unread notes'), Badge(3, secondary=True), gap='var(--space-2)')))

```html
<div stack card>
  <div cluster style="--gap: var(--space-2);">
    <h3>Field notes</h3>
<span badge>Shared</span>  </div>
  <p>Observations and ideas from the team.</p>
  <div cluster style="--gap: var(--space-2);">
<span>Unread notes</span><span badge secondary>3</span>  </div>
</div>

```

## Avatar

An avatar identifies a person or team. Give it a name and, optionally, an image URL. Without an image, it shows the first and last initials; the accessible name remains the full name. Both forms use the same circular frame. `--avatar-size` changes the frame and initials together.

In [ ]:
_avatar_css = r"""
[avatar] {
  --avatar-size: 2.5rem;
  display: inline-grid; place-items: center; vertical-align: middle; flex-shrink: 0;
  inline-size: var(--avatar-size); block-size: var(--avatar-size); border-radius: var(--radius-full);
  object-fit: cover; background: var(--v-3); color: var(--v2);
  font-size: calc(var(--avatar-size) * .4); font-weight: var(--weight1); line-height: 1;
}
"""

@hypelt(sel='[avatar]', layer='components', css=_avatar_css)
def Avatar(
    name,     # Full name, used as the accessible label and to derive initials
    src=None, # Image URL; omit to show initials
    **attrs,  # Attributes for the image or span
):
    "A circular image or initials identifying a person or team"
    if src: return Img(src=src, alt=name, avatar=True, **attrs)
    parts = name.split()
    initials = parts[0][0] if parts else '?'
    if len(parts) > 1: initials += parts[-1][0]
    return Span(initials.upper(), role='img', aria_label=name or 'Unknown', avatar=True, **attrs)

reg.rule('avatar-button', r"""
button:has(> [avatar]:only-child) {
  --pad-block: 1px; --pad-inline: 1px; border-radius: var(--radius-full); --face: var(--n-2); border-color: var(--n-1);
}
button:disabled > [avatar]:only-child { opacity: .5; }
""", layer='components', doc='A button whose only content is an avatar is round on the neutral surface, and fades its avatar when disabled.')

avatar-button [components]: A button whose only content is an avatar is round on the neutral surface, and fades its avatar when disabled.

In [ ]:
@example(Avatar)
def _avatar_example():
    return Cluster(Avatar('Ada Lovelace'), Avatar('Grace Hopper'), Avatar('Prince'),
        Avatar('Alan Turing', style='--avatar-size: 3.5rem'))

In [ ]:
Avatar.demo()

```html
<div cluster>
<span role="img" aria-label="Ada Lovelace" avatar>AL</span><span role="img" aria-label="Grace Hopper" avatar>GH</span><span role="img" aria-label="Prince" avatar>P</span><span role="img" aria-label="Alan Turing" avatar style="--avatar-size: 3.5rem">AT</span></div>

```

In [ ]:
def _picture(name): return f'https://api.dicebear.com/10.x/miniavs/svg?seed={quote(name)}&backgroundColor=ffe3ea'
demo(Cluster(Avatar('Ada Lovelace', src=_picture('Ada Lovelace'), aria_hidden='true'),
    Stack(Span('Ada Lovelace'), Small('Workspace owner'), gap='var(--space-3)')))

```html
<div cluster>
<img src="https://api.dicebear.com/10.x/miniavs/svg?seed=Ada%20Lovelace&amp;backgroundColor=ffe3ea" alt="Ada Lovelace" avatar aria-hidden="true">  <div stack style="--gap: var(--space-3);">
<span>Ada Lovelace</span><small>Workspace owner</small>  </div>
</div>

```

To make an avatar a button, we just round `Button` and put it on the neutral surface, so the avatar keeps its own colours inside a hairline ring. That is the `avatar-button` rule, registered on its own since it styles the button rather than the avatar.

In [ ]:
demo(Cluster(Button(Avatar('Ada Lovelace'), type='button'), Button(Avatar('Grace Hopper', src=_picture('Grace Hopper')), type='button'),
    Button(Avatar('Alan Turing', src=_picture('Alan Turing')), type='button', disabled=True)))

```html
<div cluster>
<button type="button"><span role="img" aria-label="Ada Lovelace" avatar>AL</span></button><button type="button"><img src="https://api.dicebear.com/10.x/miniavs/svg?seed=Grace%20Hopper&amp;backgroundColor=ffe3ea" alt="Grace Hopper" avatar></button><button type="button" disabled><img src="https://api.dicebear.com/10.x/miniavs/svg?seed=Alan%20Turing&amp;backgroundColor=ffe3ea" alt="Alan Turing" avatar></button></div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()